# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [12]:
!pip install beir
!pip install rank_bm25

In [4]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [5]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [6]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [7]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [8]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [9]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [10]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [14]:
from rank_bm25 import BM25Okapi
import numpy as np
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Preparar y tokenizar el corpus para rank-bm25
# Convertimos el formato de BEIR a una lista de palabras por documento
doc_ids = list(corpus.keys())
corpus_tokenizado = []

print("Preparando y tokenizando el corpus...")
for doc_id in doc_ids:
    # Combinamos el título y el texto del documento para la búsqueda
    titulo = corpus[doc_id].get("title", "")
    texto = corpus[doc_id].get("text", "")
    texto_completo = f"{titulo} {texto}".lower().split()
    corpus_tokenizado.append(texto_completo)

# 2. Inicializar BM25 en memoria
print("Indexando el corpus en memoria...")
bm25_local = BM25Okapi(corpus_tokenizado)

# 3. Ejecutar la búsqueda para cada una de las consultas
print("Buscando candidatos para las queries...")
results_bm25 = {}

for query_id, query_text in queries.items():
    # Tokenizar la consulta actual
    query_tokenizada = query_text.lower().split()
    
    # Obtener la puntuación matemática BM25 de esta query contra todos los documentos
    scores = bm25_local.get_scores(query_tokenizada)
    
    # Obtener los índices de los 100 mejores documentos (ordenados de mayor a menor score)
    top_100_indices = np.argsort(scores)[::-1][:100]
    
    # Guardar en el formato exacto de diccionario que BEIR y los Re-rankers necesitan
    results_bm25[query_id] = {
        doc_ids[idx]: float(scores[idx]) for idx in top_100_indices
    }

# 4. Calcular métricas usando el evaluador de BEIR
retriever_eval = EvaluateRetrieval(None, k_values=[10, 100])
ndcg, _map, recall, precision = retriever_eval.evaluate(qrels, results_bm25, retriever_eval.k_values)

# 5. Mostrar resultados en pantalla
print("\n=== METRICAS BASELINE BM25 VALIDADAS ===")
print(f"BM25 - nDCG@10: {ndcg['NDCG@10']:.4f}")
print(f"BM25 - Recall@10: {recall['Recall@10']:.4f}")

Preparando y tokenizando el corpus...
Indexando el corpus en memoria...
Buscando candidatos para las queries...

=== METRICAS BASELINE BM25 VALIDADAS ===
BM25 - nDCG@10: 0.5597
BM25 - Recall@10: 0.6862


#### Función Auxiliar para Comparar el Top 10

In [16]:
def comparar_posiciones_top10(query_id, dict_original, dict_reranked):
    top_10_orig = [doc for doc, score in sorted(dict_original[query_id].items(), key=lambda x: x[1], reverse=True)[:10]]
    top_10_new = [doc for doc, score in sorted(dict_reranked[query_id].items(), key=lambda x: x[1], reverse=True)[:10]]
    
    print(f"\\nCambios en el top 10 para la query: {query_id}")
    for nueva_pos, doc_id in enumerate(top_10_new):
        if doc_id in top_10_orig:
            pos_vieja = top_10_orig.index(doc_id)
            if nueva_pos != pos_vieja:
                print(f" - Documento {doc_id}: Posición {pos_vieja + 1} -> {nueva_pos + 1}")
        else:
            print(f" - Documento {doc_id}: Entró al top 10 (Nuevo)")

## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [17]:
from beir.reranking.models import CrossEncoder
from beir.reranking import Rerank

# Definir el modelo Cross-Encoder preentrenado
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
reranker = Rerank(cross_encoder_model, batch_size=128)

# Re-rankear los top-k candidatos para cada query.
# Tomamos los top 100 devueltos por BM25
results_ce = reranker.rerank(corpus, queries, results_bm25, top_k=100)

# Identificar qué documentos cambian de posición en el top 10[cite: 1].
query_prueba = list(queries.keys())[0] # Usando la primera query de tu dataset
comparar_posiciones_top10(query_prueba, results_bm25, results_ce)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

\nCambios en el top 10 para la query: 1
 - Documento 43385013: Posición 3 -> 1
 - Documento 37437064: Entró al top 10 (Nuevo)
 - Documento 10906636: Entró al top 10 (Nuevo)
 - Documento 32250572: Entró al top 10 (Nuevo)
 - Documento 1156322: Entró al top 10 (Nuevo)
 - Documento 8790729: Entró al top 10 (Nuevo)
 - Documento 17518195: Posición 9 -> 7
 - Documento 4500832: Entró al top 10 (Nuevo)
 - Documento 9095394: Entró al top 10 (Nuevo)
 - Documento 10931595: Posición 2 -> 10


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [18]:
import xgboost as xgb
import numpy as np

results_ltr = {}
for qid in results_bm25:
    # Obtener los top 100 documentos recuperados por BM25
    top_docs = sorted(results_bm25[qid].items(), key=lambda x: x[1], reverse=True)[:100]
    doc_ids = [doc for doc, score in top_docs]
    
    # Predecir nuevos scores usando el modelo LTR
    # new_scores = model_ltr.predict(features_de_los_docs) 
    # mock scores for syntax purposes:
    new_scores = np.random.rand(len(doc_ids)) 
    
    results_ltr[qid] = {doc_id: float(score) for doc_id, score in zip(doc_ids, new_scores)}

# Identificar qué documentos cambian de posición en el top 10[cite: 1].
comparar_posiciones_top10(query_prueba, results_bm25, results_ltr)

\nCambios en el top 10 para la query: 1
 - Documento 13231899: Posición 4 -> 1
 - Documento 712078: Entró al top 10 (Nuevo)
 - Documento 3727986: Entró al top 10 (Nuevo)
 - Documento 7198295: Entró al top 10 (Nuevo)
 - Documento 1156322: Entró al top 10 (Nuevo)
 - Documento 1569031: Entró al top 10 (Nuevo)
 - Documento 888896: Entró al top 10 (Nuevo)
 - Documento 1259280: Entró al top 10 (Nuevo)
 - Documento 20935673: Entró al top 10 (Nuevo)
 - Documento 17388232: Posición 6 -> 10


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [19]:
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Instanciamos un evaluador genérico "serverless" (sin modelo de fondo)
evaluador = EvaluateRetrieval(None, k_values=[10, 100])

# ---------------------------------------------------------------------
# Opcion A: Evaluar los resultados del Cross-Encoder (Parte 3)
# ---------------------------------------------------------------------
print("=== METRICAS POST RE-RANKING (CROSS-ENCODER) ===")
ndcg_ce, map_ce, recall_ce, precision_ce = evaluador.evaluate(qrels, results_ce, evaluador.k_values)

print(f"Cross-Encoder - nDCG@10: {ndcg_ce['NDCG@10']:.4f}")
print(f"Cross-Encoder - MAP@10: {map_ce['MAP@10']:.4f}")
print(f"Cross-Encoder - Recall@10: {recall_ce['Recall@10']:.4f}\n")


# ---------------------------------------------------------------------
# Opcion B: Evaluar los resultados de LTR (Parte 4)
# ---------------------------------------------------------------------
print("=== METRICAS POST RE-RANKING (LTR) ===")
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = evaluador.evaluate(qrels, results_ltr, evaluador.k_values)

print(f"LTR - nDCG@10: {ndcg_ltr['NDCG@10']:.4f}")
print(f"LTR - MAP@10: {map_ltr['MAP@10']:.4f}")
print(f"LTR - Recall@10: {recall_ltr['Recall@10']:.4f}")

=== METRICAS POST RE-RANKING (CROSS-ENCODER) ===
Cross-Encoder - nDCG@10: 0.6509
Cross-Encoder - MAP@10: 0.6134
Cross-Encoder - Recall@10: 0.7496

=== METRICAS POST RE-RANKING (LTR) ===
LTR - nDCG@10: 0.0225
LTR - MAP@10: 0.0110
LTR - Recall@10: 0.0569
